# Module 08 — Exercises

These exercises follow the order of the concept guide: the discrete families (Binomial, Poisson), the
continuous ones (Exponential, Normal, Beta, Gamma), then the limit theorems and Monte Carlo. Several checks
compare your answer against a *hidden* value, so compute carefully rather than guessing; `scipy.stats` is
allowed wherever the prompt does not forbid it.

**How these exercises work.** Each coding exercise has a *starter* cell (replace `# YOUR CODE HERE` with your implementation) followed by a *check* cell. Run the check cell to get an instant ✅/❌. Checks never stop the notebook, so you can run everything top-to-bottom at any time and see your score at the end. Some checks compare against a *hidden* answer (only a hash is stored), so you cannot peek at the number — think first, then verify. Conceptual questions hide their answer behind a *Show answer* toggle. Mini-projects are open-ended and come with a rubric. Difficulty is marked ★☆☆ (warm-up), ★★☆ (core), ★★★ (stretch).

### Setup and self-check helpers
Run the next two cells first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb, exp, factorial, log, sqrt, pi
from scipy import stats
rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
import hashlib as _hashlib
import numpy as _np

_results = {}

def _canon(value, decimals=3):
    """Canonical string form of an answer, so hashes are stable."""
    if isinstance(value, (bool, _np.bool_)):
        return "true" if value else "false"
    if isinstance(value, str):
        return " ".join(value.strip().lower().split())
    if isinstance(value, (int, _np.integer)):
        return str(int(value))
    if isinstance(value, (float, _np.floating)):
        s = f"{float(value):.{decimals}f}"
        return "0." + "0" * decimals if s in ("-0." + "0" * decimals, "-0") else s
    if isinstance(value, dict):
        return "{" + ",".join(f"{_canon(k, decimals)}:{_canon(v, decimals)}" for k, v in sorted(value.items(), key=lambda kv: str(kv[0]))) + "}"
    try:
        arr = _np.asarray(value)
        if arr.ndim == 0:
            return _canon(arr.item(), decimals)
        return "[" + ",".join(_canon(x, decimals) for x in arr.ravel().tolist()) + "]"
    except Exception:
        return str(value)

def sha_of(value, decimals=3):
    return _hashlib.sha256(_canon(value, decimals).encode()).hexdigest()[:16]

def check(name, condition, hint=""):
    """Record and print a pass/fail mark. Never raises, so the notebook keeps running."""
    try:
        ok = bool(condition)
    except Exception:
        ok = False
    _results[name] = ok
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"   — hint: {hint}"))
    return ok

def check_close(name, got, expected, tol=1e-6, hint=""):
    try:
        g = _np.asarray(got, dtype=float)
        e = _np.asarray(expected, dtype=float)
        ok = g.shape == e.shape and bool(_np.allclose(g, e, atol=tol, rtol=tol))
    except Exception:
        ok = False
    return check(name, ok, hint)

def check_hash(name, value, expected_sha, decimals=3, hint=""):
    """Compare an answer against a hidden expected value (only its hash is stored)."""
    try:
        ok = value is not None and sha_of(value, decimals) == expected_sha
    except Exception:
        ok = False
    return check(name, ok, hint)

def summary():
    n_ok = sum(1 for v in _results.values() if v)
    n = len(_results)
    print(f"\nScore: {n_ok}/{n} checks passed" + ("  🎉" if n and n_ok == n else ""))
    if n and n_ok < n:
        print("Not yet passing: " + ", ".join(k for k, v in _results.items() if not v))

## Exercise 1 — Binomial and Poisson probabilities with scipy.stats  `★☆☆`

A product page converts 10% of visitors; 20 visitors arrive. A help desk receives on average 3 calls per
hour. Using `scipy.stats.binom` and `scipy.stats.poisson` (methods `pmf`, `cdf`, `sf`, `ppf`), assign these
**floats** (and one **int**):

* `p_exactly_3` — exactly 3 of the 20 visitors buy;
* `p_at_most_3` — at most 3 buy;
* `p_at_least_5` — at least 5 buy (careful: `sf(k)` is $P(X > k)$, strictly greater);
* `k_95` — the smallest integer $k$ with $P(S \le k) \ge 0.95$ (an `int`);
* `p_pois_0` — no call in a given hour;
* `p_pois_ge_6` — six or more calls in an hour;
* `lam_1pct` — the hourly rate $\lambda$ at which the probability of a call-free hour would be exactly 1% (solve $e^{-\lambda} = 0.01$ by hand).

In [ ]:
p_exactly_3 = None
p_at_most_3 = None
p_at_least_5 = None
k_95 = None
p_pois_0 = None
p_pois_ge_6 = None
lam_1pct = None
# YOUR CODE HERE

**Check your answer:**

In [ ]:
check_close('P(exactly 3 of 20 buy) = C(20,3) 0.1^3 0.9^17', p_exactly_3, comb(20, 3) * 0.1**3 * 0.9**17)
check_hash('P(at most 3 buy)', p_at_most_3, '8b523e0607ff8f89', decimals=4)
check_hash('P(at least 5 buy)', p_at_least_5, '78a3cda9fa9d7f95', decimals=4)
check_hash('95% quantile of the purchase count', k_95, '4b227777d4dd1fc6')
check_close('P(no call in an hour) = e^-3', p_pois_0, exp(-3))
check_hash('P(6 or more calls)', p_pois_ge_6, '5ecc6cc295cbfd42', decimals=4)
check_hash('rate for a 1% chance of a call-free hour', lam_1pct, 'd099ef8117a2716b', decimals=3)
try:
    consistent = abs((1 - p_at_most_3) - (stats.binom.pmf(4, 20, 0.1) + p_at_least_5)) < 1e-9
except Exception:
    consistent = False
check('consistency: 1 - P(<= 3) = P(= 4) + P(>= 5)', consistent)

## Exercise 2 — Implement the Poisson PMF from scratch and watch the Binomial converge  `★☆☆`

Implement, with NumPy only (no `scipy.stats`, no `math.factorial` — which overflows for large $k$):

* `poisson_pmf(k, lam)` → $e^{-\lambda}\lambda^k/k!$ for an integer or integer array `k`; compute it in log
  space as $\exp(-\lambda + k\ln\lambda - \ln k!)$ with $\ln k! = \sum_{j=1}^{k}\ln j$ so that
  `poisson_pmf(200, 150)` works without overflow;
* `poisson_cdf(k, lam)` → $P(N \le k)$ as a float, by summing the PMF;
* `binomial_pmf(k, n, p)` → $\binom{n}{k}p^k(1-p)^{n-k}$ (here `comb` is fine).

Then compute `diff_180`, the largest absolute difference over $k = 0, \dots, 39$ between
`binomial_pmf(k, 180, 1/60)` and `poisson_pmf(k, 3)` — the guide's "180 customers each calling with
probability 1/60" example.

In [ ]:
def poisson_pmf(k, lam):
    # YOUR CODE HERE
    return None

def poisson_cdf(k, lam):
    # YOUR CODE HERE
    return None

def binomial_pmf(k, n, p):
    # YOUR CODE HERE
    return None

diff_180 = None
# YOUR CODE HERE

**Check your answer:**

In [ ]:
try:
    got = np.asarray(poisson_pmf(np.arange(0, 16), 3), dtype=float)
except Exception:
    got = None
check_close('poisson_pmf(k, 3) matches scipy for k = 0..15', got, stats.poisson.pmf(np.arange(0, 16), 3), tol=1e-9)
try:
    got = float(poisson_pmf(200, 150))
except Exception:
    got = None
check_close('poisson_pmf(200, 150) computed in log space (no overflow)', got, float(stats.poisson.pmf(200, 150)), tol=1e-12)
try:
    got = poisson_cdf(3, 3)
except Exception:
    got = None
check_hash('P(N <= 3) for lambda = 3', got, 'f123ba654da7cabb', decimals=4)
try:
    got = binomial_pmf(2, 20, 0.1)
except Exception:
    got = None
check_close('binomial_pmf(2, 20, 0.1)', got, 190 * 0.01 * 0.9**18)
check_hash('largest |Binomial(180, 1/60) - Poisson(3)| over k = 0..39', diff_180, '833fda4ddb19835e', decimals=4)
check('the two PMFs differ by less than 0.002 everywhere', diff_180 is not None and diff_180 < 0.002)

## Exercise 3 — Choose the distribution  `★☆☆`

For each quantity, name the most appropriate named distribution, its parameters (in words), and one
assumption that must hold for it to be right:

1. The number of defective screws in a box of 200 from a line with a 1.5% defect rate.
2. The number of e-mails arriving in a 10-minute window at a server that receives 300 per hour.
3. The time until the next e-mail arrives at that server.
4. The number of cold calls a salesperson makes until the first sale, if each call succeeds with probability 0.04.
5. The time until the third e-mail arrives.
6. Your uncertainty about a coin's probability of heads after seeing 12 heads in 20 flips, starting from no prior preference.
7. The class label (one of 10 digits) that an image classifier predicts, as a random variable.
8. The average of 50 independent measurements of a resistor, each with a slightly skewed error distribution.
9. The number of insurance claims per policyholder per year, when the sample variance of the counts is five times the sample mean.

<details><summary><b>Show answer</b></summary>

1. Binomial($n = 200$, $p = 0.015$) — screws must be defective independently with the same probability; since
   $n$ is large and $p$ small, Poisson($3$) is an excellent approximation.
2. Poisson($\lambda = 300 \times 10/60 = 50$) — arrivals must be independent at a constant rate (no bursts, no
   batch deliveries); if the variance of the counts exceeds the mean, use a Negative Binomial.
3. Exponential(rate $= 300$ per hour $= 5$ per minute) — same Poisson-process assumption; mean 12 seconds.
4. Geometric($p = 0.04$) on the trial number — calls independent with constant success probability; mean 25 calls.
5. Gamma(shape 3, rate 5 per minute) — the sum of three independent Exponential inter-arrival times; mean 36 seconds.
6. Beta($1 + 12, 1 + 8$) $=$ Beta(13, 9) — a Uniform (Beta(1, 1)) prior updated with 12 successes and 8
   failures; it requires the flips to be independent given the coin's bias.
7. Categorical over 10 classes with probability vector given by the softmax output; the one-hot label is
   its sample, and the multinomial would describe the class counts over many images.
8. Approximately Normal($\mu$, $\sigma^2/50$) by the central limit theorem — the measurements must be
   independent with finite variance; skewness $\gamma_1$ of a single measurement leaves a residual skewness of
   $\gamma_1/\sqrt{50}$ in the average, which is small for mild skew.
9. Negative Binomial (mean $\mu$, dispersion $r$ with $\mu + \mu^2/r = 5\mu$) — a Poisson would force variance
   $=$ mean and understate every tail probability; the heterogeneity across policyholders (a Gamma-distributed
   rate) is exactly what the Negative Binomial models.

</details>

## Exercise 4 — Exponential quantiles, tails and memorylessness  `★★☆`

Buses arrive as a Poisson process with rate $\lambda = 0.2$ per minute. Implement, with NumPy only:

* `expon_quantile(q, lam)` → the $q$-quantile of $\text{Exponential}(\lambda)$, by inverting $F(t) = 1 - e^{-\lambda t}$;
* `expon_tail(t, lam)` → $P(T > t)$;
* `conditional_tail(s, t, lam)` → $P(T > s + t \mid T > s)$ computed *from the definition* of conditional
  probability (ratio of two tail probabilities), not by assuming memorylessness;
* `sample_expon(n, lam, rng)` → `n` samples via the inverse-CDF method `-np.log(rng.random(n)) / lam`.

Then verify numerically that `conditional_tail(5, 10, 0.2)` equals `expon_tail(10, 0.2)`, and that the
empirical version — among 50,000 simulated waits longer than 5 minutes, the fraction longer than 15 — agrees.

In [ ]:
def expon_quantile(q, lam):
    # YOUR CODE HERE
    return None

def expon_tail(t, lam):
    # YOUR CODE HERE
    return None

def conditional_tail(s, t, lam):
    # YOUR CODE HERE
    return None

def sample_expon(n, lam, rng):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    got = expon_quantile(0.5, 0.2)
except Exception:
    got = None
check_hash('median waiting time (minutes)', got, '454bc3af835a6e4f', decimals=3)
try:
    got = expon_quantile(0.9, 0.2)
except Exception:
    got = None
check_hash('90th percentile of the wait', got, '488a6176fdb708a5', decimals=3)
try:
    got = expon_quantile(0.9, 0.2)
    ok_scipy = abs(got - stats.expon(scale=1/0.2).ppf(0.9)) < 1e-9
except Exception:
    ok_scipy = False
check('quantile agrees with scipy expon(scale=1/lam).ppf', ok_scipy)
try:
    got = expon_tail(10, 0.2)
except Exception:
    got = None
check_close('P(T > 10) = e^-2', got, exp(-2))
try:
    got = conditional_tail(5, 10, 0.2)
except Exception:
    got = None
check_hash('P(T > 15 | T > 5)', got, '81a66bf490ca518a', decimals=4)
try:
    memoryless = abs(conditional_tail(5, 10, 0.2) - expon_tail(10, 0.2)) < 1e-12
except Exception:
    memoryless = False
check('memorylessness: P(T > 15 | T > 5) == P(T > 10)', memoryless)
try:
    s = np.asarray(sample_expon(50_000, 0.2, np.random.default_rng(3)), dtype=float)
    ok_shape = s.shape == (50_000,) and np.all(s > 0)
    emp = float(np.mean(s[s > 5] > 15))
    ok_mean = abs(s.mean() - 5) < 0.1
except Exception:
    ok_shape, emp, ok_mean = False, None, False
check('sampler returns 50,000 positive values with mean close to 5', ok_shape and ok_mean)
check('empirical conditional tail within 0.02 of e^-2', emp is not None and abs(emp - exp(-2)) < 0.02)

## Exercise 5 — Normal z-scores and a quality-control problem  `★★☆`

A filling machine dispenses $\mathcal{N}(500, 4^2)$ grams. Implement (you may use `scipy.stats.norm`, but
all arguments must be converted to $z$-scores first — the check inspects `z_score`):

* `z_score(x, mu, sigma)` → $(x - \mu)/\sigma$;
* `p_below(x, mu, sigma)` → $P(X < x)$;
* `p_between(a, b, mu, sigma)` → $P(a < X < b)$;
* `sigma_for_tail(limit, mu, target_prob)` → the largest $\sigma$ such that $P(X < \text{limit}) \le$ `target_prob`
  (for `limit < mu`): solve $\Phi\big((\text{limit} - \mu)/\sigma\big) = $ `target_prob` for $\sigma$;
* `two_sided_interval(mu, sigma, coverage)` → the tuple $(\mu - z\sigma, \mu + z\sigma)$ containing the central
  `coverage` fraction of the mass (e.g. $z = 1.96$ for 0.95).

Questions answered by the checks: the fraction of packets below 490 g; the fraction within $\pm 5$ g; the
$\sigma$ needed so that at most 0.1% of packets fall below 490 g; the central 99% interval; and the
fraction of adults taller than 180 cm if heights are $\mathcal{N}(165, 7^2)$.

In [ ]:
def z_score(x, mu, sigma):
    # YOUR CODE HERE
    return None

def p_below(x, mu, sigma):
    # YOUR CODE HERE
    return None

def p_between(a, b, mu, sigma):
    # YOUR CODE HERE
    return None

def sigma_for_tail(limit, mu, target_prob):
    # YOUR CODE HERE
    return None

def two_sided_interval(mu, sigma, coverage):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    got = z_score(490, 500, 4)
except Exception:
    got = None
check_close('z-score of 490 g is -2.5', got, -2.5)
try:
    got = p_below(490, 500, 4)
except Exception:
    got = None
check_hash('fraction of packets below 490 g', got, '180d7f3662b77c66', decimals=4)
try:
    got = p_between(495, 505, 500, 4)
except Exception:
    got = None
check_hash('fraction within +/- 5 g', got, '28684632ee2e568b', decimals=4)
try:
    got = p_between(496, 504, 500, 4)
except Exception:
    got = None
check_close('68.27% within one sigma', got, stats.norm.cdf(1) - stats.norm.cdf(-1))
try:
    got = sigma_for_tail(490, 500, 0.001)
    ok_round_trip = abs(p_below(490, 500, got) - 0.001) < 1e-9
except Exception:
    got, ok_round_trip = None, False
check_hash('sigma needed for at most 0.1% below 490 g', got, 'ecb4142095b56ccc', decimals=3)
check('round trip: p_below(490, 500, sigma_for_tail(...)) == 0.001', ok_round_trip)
try:
    lo, hi = two_sided_interval(500, 4, 0.99)
except Exception:
    lo, hi = None, None
check_hash('central 99% interval', (lo, hi), '2db6b970f890bffc', decimals=3)
try:
    got = 1 - p_below(180, 165, 7)
except Exception:
    got = None
check_hash('fraction of adults taller than 180 cm', got, 'c863449e5baa2995', decimals=4)

## Exercise 6 — A Beta posterior for a conversion rate  `★★☆`

A landing page is believed, before any data, to convert with a rate described by a $\text{Beta}(2, 2)$ prior.
It then converts 7 of 50 visitors. Implement

* `beta_posterior(a0, b0, successes, failures)` → the tuple `(a, b)` of the posterior Beta parameters;
* `beta_mean(a, b)` and `beta_sd(a, b)` → the mean $\frac{a}{a+b}$ and standard deviation $\sqrt{\frac{ab}{(a+b)^2(a+b+1)}}$ as floats;
* `credible_interval(a, b, level=0.95)` → the tuple of the $\frac{1-\text{level}}{2}$ and $1 - \frac{1-\text{level}}{2}$ quantiles (`scipy.stats.beta.ppf`);
* `prob_b_beats_a(a_A, b_A, a_B, b_B, n_samples, rng)` → the Monte Carlo probability that a draw from
  $\text{Beta}(a_B, b_B)$ exceeds a draw from $\text{Beta}(a_A, b_A)$, using `rng.beta(a_B, b_B, n_samples)` **first** and
  `rng.beta(a_A, b_A, n_samples)` second.

Then compare the posterior mean with the raw rate $7/50$ and the prior mean $0.5$, and compute the posterior
mean you would get from 70 conversions in 500 visits (same rate, ten times the data).

In [ ]:
def beta_posterior(a0, b0, successes, failures):
    # YOUR CODE HERE
    return None

def beta_mean(a, b):
    # YOUR CODE HERE
    return None

def beta_sd(a, b):
    # YOUR CODE HERE
    return None

def credible_interval(a, b, level=0.95):
    # YOUR CODE HERE
    return None

def prob_b_beats_a(a_A, b_A, a_B, b_B, n_samples, rng):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    a, b = beta_posterior(2, 2, 7, 43)
except Exception:
    a, b = None, None
check('posterior parameters are (9, 45)', (a, b) == (9, 45))
try:
    got = beta_mean(9, 45)
except Exception:
    got = None
check_hash('posterior mean', got, 'c93110febcea19c2', decimals=4)
try:
    got = beta_sd(9, 45)
except Exception:
    got = None
check_hash('posterior standard deviation', got, 'fa1ddc57f4c4392d', decimals=4)
try:
    got = beta_sd(1, 1)
except Exception:
    got = None
check_close('sd of Beta(1, 1) = Uniform is 1/sqrt(12)', got, 1/sqrt(12))
try:
    got = credible_interval(9, 45, 0.95)
except Exception:
    got = None
check_hash('95% credible interval', got, 'fc8790774cfdd03f', decimals=3)
try:
    got = prob_b_beats_a(9, 45, 14, 40, 200_000, np.random.default_rng(11))
except Exception:
    got = None
check('P(theta_B > theta_A) is close to 0.885 (within 0.01)', got is not None and abs(got - 0.885) < 0.01)
try:
    got = beta_mean(*beta_posterior(2, 2, 70, 430))
except Exception:
    got = None
check_hash('posterior mean with ten times the data', got, '3df39ca6313dbf0f', decimals=4)
check('more data pulls the mean to within 0.01 of the raw rate 0.14', got is not None and abs(got - 0.14) < 0.01)

## Exercise 7 — Gamma waiting times and the Gamma–Poisson identity  `★★☆`

Buses arrive at rate $\lambda = 0.2$ per minute; the time until the $k$-th bus is $\text{Gamma}(k, \lambda)$ with
integer shape $k$ (an *Erlang* distribution). Implement, using NumPy and `math.factorial` only (no `scipy.stats`
inside the functions):

* `gamma_pdf(x, k, lam)` → the density $\frac{\lambda^k}{(k-1)!}x^{k-1}e^{-\lambda x}$ for $x > 0$ (0 otherwise), vectorised over `x`;
* `gamma_tail_via_poisson(t, k, lam)` → $P(\text{$k$-th bus arrives after } t)$ computed as the Poisson
  probability of **at most $k - 1$ arrivals** in $[0, t]$, $\sum_{j=0}^{k-1} e^{-\lambda t}(\lambda t)^j/j!$;
* `sample_gamma_as_sum(n, k, lam, rng)` → `n` samples as sums of `k` independent Exponentials, drawn with a single
  call `rng.exponential(1/lam, size=(n, k))` summed along axis 1.

Then confirm that `gamma_tail_via_poisson(20, 3, 0.2)` equals `scipy.stats.gamma.sf(20, a=3, scale=5)` and that the
empirical tail of 100,000 simulated third-bus times agrees.

In [ ]:
def gamma_pdf(x, k, lam):
    # YOUR CODE HERE
    return None

def gamma_tail_via_poisson(t, k, lam):
    # YOUR CODE HERE
    return None

def sample_gamma_as_sum(n, k, lam, rng):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
try:
    got = float(np.asarray(gamma_pdf(10.0, 3, 0.2)).ravel()[0])
except Exception:
    got = None
check_hash('Gamma(3, 0.2) density at x = 10', got, 'b7a80220a59886f3', decimals=4)
try:
    xs = np.linspace(0.5, 40, 80)
    got = np.asarray(gamma_pdf(xs, 3, 0.2), dtype=float)
except Exception:
    got = None
check_close('gamma_pdf matches scipy gamma.pdf on a grid', got, stats.gamma.pdf(np.linspace(0.5, 40, 80), a=3, scale=5), tol=1e-9)
try:
    got = np.asarray(gamma_pdf(np.array([-1.0, 0.0]), 3, 0.2), dtype=float)
except Exception:
    got = None
check_close('density is 0 for x <= 0', got, np.array([0.0, 0.0]))
try:
    got = gamma_tail_via_poisson(20, 3, 0.2)
except Exception:
    got = None
check_hash('P(third bus after 20 minutes)', got, '3e575a0e767f3f03', decimals=4)
check_close('Gamma tail equals Poisson CDF: matches scipy gamma.sf(20, a=3, scale=5)', got, float(stats.gamma.sf(20, a=3, scale=5)), tol=1e-9)
try:
    got = gamma_tail_via_poisson(5, 1, 0.2)
except Exception:
    got = None
check_close('k = 1 reduces to the Exponential tail e^-1', got, exp(-1))
try:
    g = np.asarray(sample_gamma_as_sum(100_000, 3, 0.2, np.random.default_rng(5)), dtype=float)
    ok = g.shape == (100_000,) and abs(g.mean() - 15) < 0.15 and abs(np.mean(g > 20) - stats.gamma.sf(20, a=3, scale=5)) < 0.01
except Exception:
    ok = False
check('simulated third-bus times: mean about 15 and tail about 0.238', ok)

## Exercise 8 — Verify the central limit theorem (and watch it fail)  `★★☆`

Implement

* `standardised_means(sampler, n, reps, mu, sigma, rng)` → a NumPy array of `reps` values
  $\sqrt{n}\,(\bar X_n - \mu)/\sigma$, where `sampler(rng, size)` returns an array of the given shape (call it
  **once** with `size=(reps, n)` and average along axis 1);
* `kolmogorov_distance(z)` → $\sup_z |F_{\text{emp}}(z) - \Phi(z)|$ between the empirical CDF of the sample and the
  standard Normal CDF (sort the sample; the sup is attained just before or at a data point — compare both
  $\frac{i}{m} - \Phi(z_{(i)})$ and $\Phi(z_{(i)}) - \frac{i-1}{m}$);
* `skewness(z)` → the sample skewness $\frac{\frac1m\sum(z_i - \bar z)^3}{\big(\frac1m\sum(z_i - \bar z)^2\big)^{3/2}}$.

Then test the CLT: for an Exponential(1) source ($\mu = \sigma = 1$) with $n = 30$ and 20,000 repetitions the
standardised means should have mean $\approx 0$, variance $\approx 1$, Kolmogorov distance below 0.04 and skewness
near $2/\sqrt{30} = 0.365$. For a Cauchy source (use $\mu = 0$, $\sigma = 1$ as *nominal* values) the variance
of the "standardised means" should blow up and the Kolmogorov distance should stay large for every $n$.

In [ ]:
def standardised_means(sampler, n, reps, mu, sigma, rng):
    # YOUR CODE HERE
    return None

def kolmogorov_distance(z):
    # YOUR CODE HERE
    return None

def skewness(z):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
expo = lambda rng, size: rng.exponential(1.0, size)
try:
    z = np.asarray(standardised_means(expo, 30, 20_000, 1.0, 1.0, np.random.default_rng(1)), dtype=float)
    ok_shape = z.shape == (20_000,)
except Exception:
    z, ok_shape = None, False
check('returns 20,000 standardised means', ok_shape)
check('mean of standardised means is within 0.03 of 0', ok_shape and abs(z.mean()) < 0.03)
check('variance of standardised means is within 0.05 of 1', ok_shape and abs(z.var() - 1) < 0.05)
try:
    kd = kolmogorov_distance(z)
except Exception:
    kd = None
check('Kolmogorov distance to N(0,1) below 0.04 for exponential means of 30', kd is not None and kd < 0.04)
try:
    kd_normal = kolmogorov_distance(stats.norm.ppf((np.arange(1, 1001) - 0.5) / 1000))
except Exception:
    kd_normal = None
check_close('Kolmogorov distance of perfect Normal quantiles is 0.0005', kd_normal, 0.0005, tol=1e-9)
try:
    kd1 = kolmogorov_distance(standardised_means(expo, 1, 20_000, 1.0, 1.0, np.random.default_rng(2)))
except Exception:
    kd1 = None
check('n = 1 (no averaging): distance is large, about 0.16 (within 0.02)', kd1 is not None and abs(kd1 - 0.1587) < 0.02)
try:
    sk = skewness(z)
except Exception:
    sk = None
check('skewness of the means is near 2/sqrt(30) = 0.365 (within 0.08)', sk is not None and abs(sk - 2/sqrt(30)) < 0.08)
try:
    got = skewness(np.array([1.0, 2.0, 3.0, 4.0, 10.0]))
except Exception:
    got = None
check_hash('skewness of [1, 2, 3, 4, 10]', got, '9fc3c77c94302ac0', decimals=4)
try:
    cauchy = lambda rng, size: rng.standard_cauchy(size)
    zc = standardised_means(cauchy, 1000, 20_000, 0.0, 1.0, np.random.default_rng(3))
    ok_cauchy = np.var(zc) > 100 and kolmogorov_distance(zc) > 0.2
except Exception:
    ok_cauchy = False
check('Cauchy source: variance explodes and the distance to the Normal stays large', ok_cauchy)

## Exercise 9 — Diagnose the argument: 'the law of large numbers guarantees it'  `★★☆`

Three colleagues each invoke the law of large numbers. For each, say whether the conclusion follows, and if
not, name the error precisely and give the correct statement.

1. "Our roulette table has paid out more than expected this week. By the law of large numbers, next week it
   will pay out *less* than expected so that the long-run average is restored."
2. "Our fraud model's per-transaction loss has a heavy tail — a few transactions cost millions. But we
   average the loss over 10 million transactions a day, so by the law of large numbers the daily average is
   essentially constant and we can budget on it."
3. "We evaluate on 200 test examples. By the central limit theorem the accuracy estimate is Normal, so its
   95% interval is $\hat p \pm 1.96\sqrt{\hat p(1-\hat p)/200}$ — fine for our model with $\hat p = 0.995$."

<details><summary><b>Show answer</b></summary>

1. Gambler's fallacy. The LLN says the *proportion* $\bar X_n$ converges; it says nothing about the *sum*
   $S_n - n\mu$, which does not revert and in fact wanders further (its standard deviation grows like
   $\sqrt{n}$). Next week's spins are independent of this week's and have the same expectation; the excess is
   diluted by future spins, not repaid. Correct statement: the fraction of the long-run total that this week
   represents shrinks toward zero.
2. Two problems. First, the LLN needs a finite *mean*, and the rate of convergence needs a finite *variance*:
   if the loss tail is a power law with index $\alpha \le 2$, the daily average has infinite variance, is
   dominated by the single largest loss, and does not stabilise like $\sigma/\sqrt{n}$ (if $\alpha \le 1$ it does
   not converge at all). Second, the 10 million transactions are not independent (fraud rings, system outages
   produce clusters), so the effective $n$ is far smaller. Correct approach: model the tail explicitly
   (generalised Pareto), budget on a high quantile of the daily total, and reinsure or cap exposures.
3. The CLT is asymptotic and its speed depends on skewness: with $p = 0.995$ and $n = 200$ the expected number of
   errors is just 1, the Bernoulli skewness is $(1 - 2p)/\sqrt{p(1-p)} \approx -14$, and the sampling distribution
   of $\hat p$ is a few discrete spikes, not a bell. The Wald interval $\hat p \pm 1.96\sqrt{\hat p(1-\hat p)/n}$
   is badly miscalibrated here (if 0 errors are observed it collapses to a point). Correct approach: use an exact
   Binomial (Clopper–Pearson) or Wilson interval, which for 1 error in 200 gives roughly $[0.97, 1.0]$ — far
   wider than the Wald interval suggests — and recognise that distinguishing 99.5% from 99% needs thousands of
   examples, not 200.

</details>

## Exercise 10 — Monte Carlo integration with an error bar, plus antithetic variates  `★★★`

Implement

* `mc_integral(f, a, b, n, rng)` → the tuple `(estimate, standard_error)` for $\int_a^b f(x)\,dx$, using
  `n` uniform points `x = a + (b - a) * rng.random(n)`: the estimate is the mean of $(b-a)f(x_i)$ and the standard
  error is their sample standard deviation (`ddof=1`) divided by $\sqrt{n}$;
* `mc_integral_antithetic(f, a, b, n, rng)` → the same with **antithetic variates**: draw `u = rng.random(n // 2)`,
  evaluate $f$ at both $a + (b-a)u$ and $a + (b-a)(1-u)$, average each pair into one value
  $\frac{b-a}{2}[f(x_1) + f(x_2)]$, and return the mean and standard error (`ddof=1`, divided by $\sqrt{n/2}$)
  of those $n/2$ pair values;
* `samples_needed(sigma, target_se)` → the smallest integer $n$ with $\sigma/\sqrt{n} \le$ `target_se`.

Targets: $\int_0^1 e^{-x^2}\,dx = 0.746824$ and $\int_1^5 \frac{\sin x}{x}\,dx = 0.603848$. Your estimates must be
within 4 standard errors of the truth, and the antithetic standard error for $e^{-x^2}$ must be at least 3 times
smaller than the plain one (the guide predicts about 5 times).

In [ ]:
def mc_integral(f, a, b, n, rng):
    # YOUR CODE HERE
    return None

def mc_integral_antithetic(f, a, b, n, rng):
    # YOUR CODE HERE
    return None

def samples_needed(sigma, target_se):
    # YOUR CODE HERE
    return None

**Check your answer:**

In [ ]:
I1, I2 = 0.7468241328124271, 0.6038481290459576
f1 = lambda x: np.exp(-x**2)
f2 = lambda x: np.sin(x) / x
try:
    est, se = mc_integral(f1, 0.0, 1.0, 10_000, np.random.default_rng(7))
except Exception:
    est, se = None, None
check('plain estimate of int_0^1 exp(-x^2) within 4 SE of 0.746824', est is not None and se is not None and abs(est - I1) < 4 * se)
check('its standard error is about 0.002 (within 0.0005)', se is not None and abs(se - 0.00201) < 0.0005)
check_hash('seeded plain estimate (rng seed 7, n = 10,000)', est, '8e6adeb6703ed174', decimals=4)
try:
    est2, se2 = mc_integral(f2, 1.0, 5.0, 20_000, np.random.default_rng(8))
except Exception:
    est2, se2 = None, None
check('estimate of int_1^5 sin(x)/x within 4 SE of 0.603848', est2 is not None and se2 is not None and abs(est2 - I2) < 4 * se2)
check('the (b - a) factor is applied (estimate is not 0.15)', est2 is not None and est2 > 0.5)
try:
    est_a, se_a = mc_integral_antithetic(f1, 0.0, 1.0, 10_000, np.random.default_rng(7))
except Exception:
    est_a, se_a = None, None
check('antithetic estimate within 4 SE of the truth', est_a is not None and se_a is not None and abs(est_a - I1) < 4 * se_a)
check('antithetic SE at least 3x smaller than plain SE', se is not None and se_a is not None and se_a > 0 and se / se_a > 3)
check_hash('seeded antithetic estimate', est_a, '2755ec8f98da7bf1', decimals=4)
try:
    got = samples_needed(0.201, 0.0002)
except Exception:
    got = None
check_hash('samples needed for SE 0.0002 with sigma 0.201', got, 'ee50b7c98e41e161')
try:
    got = samples_needed(1.0, 0.1)
except Exception:
    got = None
check_close('samples_needed(1, 0.1) = 100', got, 100)

## Exercise 11 — Mini-project: staffing a call centre with Poisson arrivals and Exponential service  `★★★`

A call centre receives calls as a Poisson process at **2 calls per minute** during an 8-hour day; each call
occupies one agent for an **Exponential** time with mean **4 minutes**. Calls that find every agent busy wait in
a single first-come-first-served queue. Build a simulation and answer the manager's question: *how many
agents are needed so that at least 90% of calls are answered within 20 seconds?*

**The model** (use `rng = np.random.default_rng(2024)`):

1. Generate arrival times as the cumulative sum of Exponential(rate 2) gaps over the day (add a one-hour warm-up
   whose calls you discard, so the empty start of the day does not bias the results); generate an Exponential
   service time for each call.
2. Keep an array `free_at` with the time each of the $c$ agents becomes free. Process calls in arrival order:
   the earliest-free agent takes the call, it starts at `max(arrival, free_at[agent])`, the wait is the
   difference, and the agent is busy until `start + service`.
3. Simulate 60 days for each $c$ from 9 to 15 agents and pool the waiting times.

**Deliverables**

* a table over $c$ of the occupancy (offered load $2 \times 4 = 8$ divided by $c$), the fraction of calls that wait at
  all, the service level $P(\text{wait} \le 20\text{ s})$, and the mean, 90th and 99th percentile waiting times;
* the smallest $c$ meeting the 90% target, and a histogram of waiting times for that $c$ and its neighbours;
* a sanity check that the arrival gaps are memoryless, and a comparison of your simulated service level with the
  analytic **Erlang C** formula for the $M/M/c$ queue:
  $P(\text{wait} > 0) = \frac{A^c/c! \cdot c/(c-A)}{\sum_{k<c} A^k/k! + A^c/c! \cdot c/(c-A)}$ with $A = \lambda/\mu = 8$,
  and $P(\text{wait} > t) = P(\text{wait} > 0)\,e^{-(c\mu - \lambda)t}$;
* a standard error for the service-level estimate computed from the **day-to-day** variation of 60 daily service
  levels, compared with the naive $\sqrt{p(1-p)/n}$ over all pooled calls — explain why they differ;
* a sensitivity check: the service level at your chosen $c$ if the call rate rises by 10%.

**Rubric — you have mastered this when you can:**

* Arrivals are generated as a Poisson process (Exponential gaps, cumulative sum) and service times as Exponentials,
  all from one seeded generator; a warm-up period is discarded.
* The queue is simulated correctly: the earliest-free agent serves the next call, waits are `start - arrival`, and
  a call that finds a free agent has wait 0.
* The table reports occupancy, $P(\text{wait} > 0)$, the service level and waiting-time quantiles for each staffing
  level, and the smallest $c$ meeting the 90%-within-20-seconds target is identified (12 agents).
* The simulated service levels agree with the Erlang C formula to within about 0.01 for every $c$.
* The histogram has a title, axis labels and a legend, and shows how one agent more or fewer changes the
  waiting-time distribution.
* The standard error is computed from independent simulated days, and the answer explains that pooled calls are
  positively correlated (a busy spell delays many calls at once), so $\sqrt{p(1-p)/n}$ over calls understates the
  uncertainty.

In [ ]:
# Your mini-project workspace

## Score

In [ ]:
summary()